# Grid diagnostics: models & observations

This notebook checks:

1. **Model outputs** (`data/model_1m`)  
   - Whether all models share the same lat/lon grid.  
   - Time coverage and monthly spacing.

2. **Observed datasets** (`data/observed_1m`)  
   - Whether each product matches the **canonical ISIMIP land mask grid**  
     (no Antarctica, no Greenland, 0.5° resolution).  
   - Time coverage and monthly spacing.

The summaries at the end of each section highlight any inconsistencies.

## Imports, registry, shared helpers

In [8]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from sm_attribution.io.registry import default_registry
from sm_attribution.io.load_mask import load_isimip_landmask

reg = default_registry()


def _sha1(arr: np.ndarray) -> str:
    """Return a SHA-1 hash of a numpy array's raw bytes (for coord fingerprints)."""
    a = np.asarray(arr)
    return hashlib.sha1(a.tobytes()).hexdigest()


def _monthly_spacing_ok(time_idx) -> bool:
    """
    Check that a time index is strictly monthly with constant 1-month steps.
    For short series (<3) we consider it OK by default.
    """
    if time_idx.size < 3:
        return True
    diffs = np.diff(time_idx.values.astype("datetime64[M]").astype("int64"))
    return np.all(diffs == 1)


def _lon_wrap_to_180(lon: np.ndarray) -> np.ndarray:
    """
    Return longitude in [-180, 180) for diagnostics, without modifying
    the underlying dataset.
    """
    if np.nanmax(lon) > 180.0:
        lon = ((lon + 180.0) % 360.0) - 180.0
    return lon

# Pandas display options for full DataFrame visibility
pd.set_option("display.max_rows", None)        # show all rows
pd.set_option("display.max_columns", None)     # show all columns
pd.set_option("display.width", None)           # don't wrap columns
pd.set_option("display.max_colwidth", None)    # show long text fully

## Model grid diagnostics

In [9]:
# Scenarios and models
SCENARIOS = reg.scenarios()
MODELS = [
    "h08",
    "hydropy",
    "jules-w2",
    "miroc-integ-land",
    "watergap2-2e",
    "web-dhm-sg",
    "lpjml5-7-10-fire",
]


def read_model_1m(model: str, scenario: str):
    """
    Load a processed model_1m file and return the main soil moisture variable
    as a (time, lat, lon) DataArray plus its file path.
    """
    path = reg.get_model_processed(model, scenario)
    ds = xr.open_dataset(path, decode_times=True)

    # Normalize coordinate names
    ren = {}
    if "latitude" in ds.coords:
        ren["latitude"] = "lat"
    if "longitude" in ds.coords:
        ren["longitude"] = "lon"
    if ren:
        ds = ds.rename(ren)

    # Prefer 'soilmoist_1m' if present, otherwise fall back to the first data var
    var = "soilmoist_1m" if "soilmoist_1m" in ds.data_vars else list(ds.data_vars)[0]
    da = ds[var]

    # Ensure consistent dimension ordering for diagnostics
    da = da.transpose("time", "lat", "lon", missing_dims="ignore")
    return path, da


rows = []
canon_lat = canon_lon = None
canon_src = None

for m in MODELS:
    for s in SCENARIOS:
        path, da = read_model_1m(m, s)

        has_lat = "lat" in da.coords
        has_lon = "lon" in da.coords
        has_time = "time" in da.coords

        lat = da["lat"].values.copy() if has_lat else np.array([])
        lon = da["lon"].values.copy() if has_lon else np.array([])

        lon_wrapped = _lon_wrap_to_180(lon)

        # Use the first encountered model/scenario grid as canonical
        if canon_lat is None or canon_lon is None:
            canon_lat = lat.copy()
            canon_lon = lon_wrapped.copy()
            canon_src = f"{m}_{s} (first encountered)"

        same_len = (lat.size == canon_lat.size) and (lon_wrapped.size == canon_lon.size)
        same_lat_vals = same_len and np.array_equal(lat, canon_lat)
        same_lon_vals = same_len and np.array_equal(lon_wrapped, canon_lon)

        # Monotonicity and step
        if lat.size > 1:
            dlat = np.diff(lat)
            lat_monot = np.all(dlat > 0) or np.all(dlat < 0)
            lat_step = np.median(dlat)
        else:
            lat_monot = True
            lat_step = np.nan

        if lon_wrapped.size > 1:
            dlon = np.diff(lon_wrapped)
            lon_monot = np.all(dlon > 0) or np.all(dlon < 0)
            lon_step = np.median(dlon)
        else:
            lon_monot = True
            lon_step = np.nan

        # Time info
        cal = da["time"].attrs.get("calendar", "") if has_time else ""
        tmin = pd.to_datetime(str(da["time"].values[0])) if has_time else None
        tmax = pd.to_datetime(str(da["time"].values[-1])) if has_time else None
        monthly_ok = _monthly_spacing_ok(da["time"]) if has_time else True

        # Max absolute deviations vs canonical (helpful for small shifts)
        lat_max_absdiff = np.max(np.abs(lat - canon_lat)) if same_len else np.nan
        lon_max_absdiff = np.max(np.abs(lon_wrapped - canon_lon)) if same_len else np.nan

        rows.append(
            dict(
                model=m,
                scenario=s,
                path=path,
                lat_len=lat.size,
                lon_len=lon.size,
                lat_min=np.nanmin(lat) if lat.size else np.nan,
                lat_max=np.nanmax(lat) if lat.size else np.nan,
                lon_min=np.nanmin(lon_wrapped) if lon_wrapped.size else np.nan,
                lon_max=np.nanmax(lon_wrapped) if lon_wrapped.size else np.nan,
                lat_step=float(lat_step),
                lon_step=float(lon_step),
                lat_monot=bool(lat_monot),
                lon_monot=bool(lon_monot),
                same_len=bool(same_len),
                same_lat_vals=bool(same_lat_vals),
                same_lon_vals=bool(same_lon_vals),
                lat_max_absdiff=float(lat_max_absdiff) if same_len else np.nan,
                lon_max_absdiff=float(lon_max_absdiff) if same_len else np.nan,
                time_len=int(da.sizes["time"]) if has_time else 0,
                time_min=str(tmin) if tmin is not None else "",
                time_max=str(tmax) if tmax is not None else "",
                calendar=cal,
                monthly_spacing_ok=bool(monthly_ok),
                lat_hash=_sha1(lat),
                lon_hash=_sha1(lon_wrapped),
            )
        )

df_model = (
    pd.DataFrame(rows).sort_values(["model", "scenario"]).reset_index(drop=True)
)

print(f"Canonical model grid source: {canon_src}")
display(df_model)

print("\n--- ANY coord issues vs model canonical? ---")
print(
    df_model.loc[
        ~df_model["same_len"] | ~df_model["lat_monot"] | ~df_model["lon_monot"],
        ["model", "scenario", "lat_len", "lon_len", "lat_monot", "lon_monot"],
    ]
)

print("\n--- ANY lat/lon mismatches vs model canonical? ---")
print(
    df_model.loc[
        (~df_model["same_lat_vals"]) | (~df_model["same_lon_vals"]),
        ["model", "scenario", "lat_max_absdiff", "lon_max_absdiff", "lat_step", "lon_step"],
    ]
)

print("\n--- TIME diagnostics (models) ---")
print(
    df_model.loc[
        ~df_model["monthly_spacing_ok"] | (df_model["calendar"] != "proleptic_gregorian"),
        ["model", "scenario", "time_len", "time_min", "time_max", "calendar", "monthly_spacing_ok"],
    ]
)

Canonical model grid source: h08_obsclim_histsoc (first encountered)


,model,scenario,path,lat_len,lon_len,lat_min,lat_max,lon_min,lon_max,lat_step,lon_step,lat_monot,lon_monot,same_len,same_lat_vals,same_lon_vals,lat_max_absdiff,lon_max_absdiff,time_len,time_min,time_max,calendar,monthly_spacing_ok,lat_hash,lon_hash
0,h08,counterclim_1901soc,/Users/thchilly/projects/sm_attribution/data/models_1m/h08_gswp3-w5e5_counterclim_1901soc_default_soilmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
1,h08,counterclim_histsoc,/Users/thchilly/projects/sm_attribution/data/models_1m/h08_gswp3-w5e5_counterclim_histsoc_default_soilmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
2,h08,obsclim_1901soc,/Users/thchilly/projects/sm_attribution/data/models_1m/h08_gswp3-w5e5_obsclim_1901soc_default_soilmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
3,h08,obsclim_histsoc,/Users/thchilly/projects/sm_attribution/data/models_1m/h08_gswp3-w5e5_obsclim_histsoc_default_soilmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
4,hydropy,counterclim_1901soc,/Users/thchilly/projects/sm_attribution/data/models_1m/hydropy_gswp3-w5e5_counterclim_1901soc_default_rootmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
5,hydropy,counterclim_histsoc,/Users/thchilly/projects/sm_attribution/data/models_1m/hydropy_gswp3-w5e5_counterclim_histsoc_default_rootmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
6,hydropy,obsclim_1901soc,/Users/thchilly/projects/sm_attribution/data/models_1m/hydropy_gswp3-w5e5_obsclim_1901soc_default_rootmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
7,hydropy,obsclim_histsoc,/Users/thchilly/projects/sm_attribution/data/models_1m/hydropy_gswp3-w5e5_obsclim_histsoc_default_rootmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
8,jules-w2,counterclim_1901soc,/Users/thchilly/projects/sm_attribution/data/models_1m/jules-w2_gswp3-w5e5_counterclim_1901soc_default_soilmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
9,jules-w2,counterclim_histsoc,/Users/thchilly/projects/sm_attribution/data/models_1m/jules-w2_gswp3-w5e5_counterclim_histsoc_default_soilmoist_global_monthly_1901_2019_1m.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,True,True,True,True,True,0.0,0.0,1428,1901-01-01 00:00:00,2019-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb52


--- ANY coord issues vs model canonical? ---
Empty DataFrame
Columns: [model, scenario, lat_len, lon_len, lat_monot, lon_monot]
Index: []

--- ANY lat/lon mismatches vs model canonical? ---
Empty DataFrame
Columns: [model, scenario, lat_max_absdiff, lon_max_absdiff, lat_step, lon_step]
Index: []

--- TIME diagnostics (models) ---
               model             scenario  time_len             time_min  \
0                h08  counterclim_1901soc      1428  1901-01-01 00:00:00   
1                h08  counterclim_histsoc      1428  1901-01-01 00:00:00   
2                h08      obsclim_1901soc      1428  1901-01-01 00:00:00   
3                h08      obsclim_histsoc      1428  1901-01-01 00:00:00   
4            hydropy  counterclim_1901soc      1428  1901-01-01 00:00:00   
5            hydropy  counterclim_histsoc      1428  1901-01-01 00:00:00   
6            hydropy      obsclim_1901soc      1428  1901-01-01 00:00:00   
7            hydropy      obsclim_histsoc      1428  1901-0

## Canonical ISIMIP mask & quick model sanity check

In [3]:
# Canonical land mask (no Antarctica, no Greenland) for 0.5° grid checks
land = load_isimip_landmask("isimip_no_ant_nogreenland")
canon_lat_mask = land["lat"].values.copy()
canon_lon_mask = land["lon"].values.copy()

print("MASK dims:", land.dims, {k: int(land.sizes[k]) for k in land.dims})
print("MASK lat min/max:", canon_lat_mask.min(), canon_lat_mask.max())
print("MASK lon min/max:", canon_lon_mask.min(), canon_lon_mask.max())

# Optional: sanity check that one model matches the mask grid exactly
m_path = reg.get_model_processed("h08", "obsclim_histsoc")
ds_m = xr.open_dataset(m_path)
if "latitude" in ds_m.coords:
    ds_m = ds_m.rename({"latitude": "lat"})
if "longitude" in ds_m.coords:
    ds_m = ds_m.rename({"longitude": "lon"})

lat_m = ds_m["lat"].values
lon_m = _lon_wrap_to_180(ds_m["lon"].values)

print("MODEL(h08, obsclim_histsoc) lat match mask?", np.array_equal(lat_m, canon_lat_mask))
print("MODEL(h08, obsclim_histsoc) lon match mask?", np.array_equal(lon_m, canon_lon_mask))

MASK dims: ('lat', 'lon') {'lat': 360, 'lon': 720}
MASK lat min/max: -89.75 89.75
MASK lon min/max: -179.75 179.75
MODEL(h08, obsclim_histsoc) lat match mask? True
MODEL(h08, obsclim_histsoc) lon match mask? True


/Users/thchilly/projects/sm_attribution/src/sm_attribution/io/load_mask.py:21: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_dataset(path)


## Observations diagnostics vs ISIMIP mask

In [10]:
def read_obs_1m(obs_key: str):
    """
    Load an observed_1m dataset from the registry and return a (time, lat, lon) DataArray.
    Picks the first variable that has time/lat/lon dims.
    """
    path = reg.get_obs_processed(obs_key)
    ds = xr.open_dataset(path, decode_times=True)

    # Normalize coord names
    ren = {}
    if "latitude" in ds.coords:
        ren["latitude"] = "lat"
    if "longitude" in ds.coords:
        ren["longitude"] = "lon"
    ds = ds.rename(ren)

    # Candidate variables with (time, lat, lon) dims
    candidates = [
        k for k in ds.data_vars
        if {"time", "lat", "lon"}.issubset(set(ds[k].dims))
    ]
    var = candidates[0] if candidates else list(ds.data_vars)[0]

    da = ds[var]
    da = da.transpose("time", "lat", "lon", missing_dims="ignore")
    return path, da


OBS_KEYS = [
    "era5land_1950_2020",
    "gleam42a_1980_2020",
    "gleam42a_2003_2020",
    "gleam42b_2003_2020",
    "gldas_v20_1948_2014",
    "gldas_v21_2000_2020",
    "gracedadm_2003_2020",
    "somo_ml_0p5m_2000_2019",
    "merra2_1980_2020",
    "gdo_smia_1995_2020",
    "gdo_ensmia_2001_2020",
]

rows = []

for obs_key in OBS_KEYS:
    path, da = read_obs_1m(obs_key)

    has_lat = "lat" in da.coords
    has_lon = "lon" in da.coords
    has_time = "time" in da.coords

    lat = da["lat"].values.copy() if has_lat else np.array([])
    lon = da["lon"].values.copy() if has_lon else np.array([])

    lon_wrapped = _lon_wrap_to_180(lon)

    # Compare to canonical ISIMIP mask grid
    same_len = (lat.size == canon_lat_mask.size) and (lon_wrapped.size == canon_lon_mask.size)
    same_lat_vals = same_len and np.array_equal(lat, canon_lat_mask)
    same_lon_vals = same_len and np.array_equal(lon_wrapped, canon_lon_mask)

    lat_monot = np.all(np.diff(lat) > 0) if lat.size > 1 else True
    lon_monot = np.all(np.diff(lon_wrapped) > 0) if lon_wrapped.size > 1 else True
    lat_step = np.median(np.diff(lat)) if lat.size > 1 else np.nan
    lon_step = np.median(np.diff(lon_wrapped)) if lon_wrapped.size > 1 else np.nan

    # Time checks
    cal = da["time"].attrs.get("calendar", "") if has_time else ""
    tmin = pd.to_datetime(str(da["time"].values[0])) if has_time else None
    tmax = pd.to_datetime(str(da["time"].values[-1])) if has_time else None
    monthly_ok = _monthly_spacing_ok(da["time"]) if has_time else True

    lat_max_absdiff = np.max(np.abs(lat - canon_lat_mask)) if same_len else np.nan
    lon_max_absdiff = np.max(np.abs(lon_wrapped - canon_lon_mask)) if same_len else np.nan

    rows.append(
        dict(
            obs_key=obs_key,
            path=path,
            lat_len=lat.size,
            lon_len=lon.size,
            lat_min=np.nanmin(lat) if lat.size else np.nan,
            lat_max=np.nanmax(lat) if lat.size else np.nan,
            lon_min=np.nanmin(lon_wrapped) if lon_wrapped.size else np.nan,
            lon_max=np.nanmax(lon_wrapped) if lon_wrapped.size else np.nan,
            lat_step=float(lat_step),
            lon_step=float(lon_step),
            lat_monot=bool(lat_monot),
            lon_monot=bool(lon_monot),
            same_len=bool(same_len),
            same_lat_vals=bool(same_lat_vals),
            same_lon_vals=bool(same_lon_vals),
            lat_max_absdiff=float(lat_max_absdiff) if same_len else np.nan,
            lon_max_absdiff=float(lon_max_absdiff) if same_len else np.nan,
            time_len=int(da.sizes["time"]) if has_time else 0,
            time_min=str(tmin) if tmin is not None else "",
            time_max=str(tmax) if tmax is not None else "",
            calendar=cal,
            monthly_spacing_ok=bool(monthly_ok),
            lat_hash=_sha1(lat),
            lon_hash=_sha1(lon_wrapped),
        )
    )

df_obs = pd.DataFrame(rows).sort_values("obs_key").reset_index(drop=True)

print("Canonical grid from:", "ISIMIP mask (no-ant, no Greenland)")
display(df_obs)

print("\n--- ANY coord issues vs mask? ---")
display(
    df_obs.loc[
        ~df_obs["same_len"] | ~df_obs["lat_monot"] | ~df_obs["lon_monot"],
        ["obs_key", "lat_len", "lon_len", "lat_monot", "lon_monot"],
    ]
)

print("\n--- ANY lat/lon mismatches vs canonical mask? ---")
display(
    df_obs.loc[
        (~df_obs["same_lat_vals"]) | (~df_obs["same_lon_vals"]),
        ["obs_key", "lat_max_absdiff", "lon_max_absdiff", "lat_step", "lon_step"],
    ]
)

print("\n--- TIME diagnostics (obs) ---")
display(
    df_obs.loc[
        ~df_obs["monthly_spacing_ok"] | (df_obs["calendar"] != "proleptic_gregorian"),
        ["obs_key", "time_len", "time_min", "time_max", "calendar", "monthly_spacing_ok"],
    ]
)

Canonical grid from: ISIMIP mask (no-ant, no Greenland)


,obs_key,path,lat_len,lon_len,lat_min,lat_max,lon_min,lon_max,lat_step,lon_step,lat_monot,lon_monot,same_len,same_lat_vals,same_lon_vals,lat_max_absdiff,lon_max_absdiff,time_len,time_min,time_max,calendar,monthly_spacing_ok,lat_hash,lon_hash
0,era5land_1950_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/ERA5-LAND/ERA5L_soilmoist_1m_monthly_0p5deg_1950_2020.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,852,1950-01-01 00:00:00,2020-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
1,gdo_ensmia_2001_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/GDO-ENSMIA/GDO_ENSMIA_soilmoist_anom_std_monthly_0p5deg_2001_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,240,2001-01-01 00:00:00,2020-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
2,gdo_smia_1995_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/GDO-SMIA/gdo_smia_soilmoist_anom_std_monthly_0p5deg_1995_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,312,1995-01-01 00:00:00,2020-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
3,gldas_v20_1948_2014,/Users/thchilly/projects/sm_attribution/data/observed_1m/GLDAS/GLDAS_NOAH_v2.0_soilmoist_1m_monthly_0p5deg_1948_2014_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,803,1948-02-01 00:00:00,2014-12-01 00:00:00,,True,279f458b7f0075ea76016fa37c23b5a50262c558,f9637d66c33ab5e892ad90963c82d4cbceb72480
4,gldas_v21_2000_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/GLDAS/GLDAS_NOAH_v2.1_soilmoist_1m_monthly_0p5deg_2000_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,252,2000-01-01 00:00:00,2020-12-01 00:00:00,,True,279f458b7f0075ea76016fa37c23b5a50262c558,f9637d66c33ab5e892ad90963c82d4cbceb72480
5,gleam42a_1980_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/GLEAM/GLEAM_v4.2a_SMrz_monthly_0p5deg_1980_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,492,1980-01-01 00:00:00,2020-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
6,gleam42a_2003_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/GLEAM/GLEAM_v4.2a_SMrz_monthly_0p5deg_2003_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,216,2003-01-01 00:00:00,2020-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
7,gleam42b_2003_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/GLEAM/GLEAM_v4.2b_SMrz_monthly_0p5deg_2003_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,216,2003-01-01 00:00:00,2020-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34
8,gracedadm_2003_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/GRACEDADM/GRACEDADM_CLSM025GL_rootzone_percentile_monthly_0p5deg_2003_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,215,2003-02-01 00:00:00,2020-12-01 00:00:00,,True,279f458b7f0075ea76016fa37c23b5a50262c558,f9637d66c33ab5e892ad90963c82d4cbceb72480
9,merra2_1980_2020,/Users/thchilly/projects/sm_attribution/data/observed_1m/MERRA2-LAND/MERRA2L_soilmoist_1m_monthly_0p5deg_1980_2020_v0.nc,360,720,-89.75,89.75,-179.75,179.75,-0.5,0.5,False,True,True,True,True,0.0,0.0,492,1980-01-01 00:00:00,2020-12-01 00:00:00,,True,6a1f7788d6c78346a1c2221246b9ed934d6de851,f6cfa44fbb5254e918f97990cbc4896d88c56f34



--- ANY coord issues vs mask? ---


,obs_key,lat_len,lon_len,lat_monot,lon_monot
0,era5land_1950_2020,360,720,False,True
1,gdo_ensmia_2001_2020,360,720,False,True
2,gdo_smia_1995_2020,360,720,False,True
3,gldas_v20_1948_2014,360,720,False,True
4,gldas_v21_2000_2020,360,720,False,True
5,gleam42a_1980_2020,360,720,False,True
6,gleam42a_2003_2020,360,720,False,True
7,gleam42b_2003_2020,360,720,False,True
8,gracedadm_2003_2020,360,720,False,True
9,merra2_1980_2020,360,720,False,True



--- ANY lat/lon mismatches vs canonical mask? ---


,obs_key,lat_max_absdiff,lon_max_absdiff,lat_step,lon_step



--- TIME diagnostics (obs) ---


,obs_key,time_len,time_min,time_max,calendar,monthly_spacing_ok
0,era5land_1950_2020,852,1950-01-01 00:00:00,2020-12-01 00:00:00,,True
1,gdo_ensmia_2001_2020,240,2001-01-01 00:00:00,2020-12-01 00:00:00,,True
2,gdo_smia_1995_2020,312,1995-01-01 00:00:00,2020-12-01 00:00:00,,True
3,gldas_v20_1948_2014,803,1948-02-01 00:00:00,2014-12-01 00:00:00,,True
4,gldas_v21_2000_2020,252,2000-01-01 00:00:00,2020-12-01 00:00:00,,True
5,gleam42a_1980_2020,492,1980-01-01 00:00:00,2020-12-01 00:00:00,,True
6,gleam42a_2003_2020,216,2003-01-01 00:00:00,2020-12-01 00:00:00,,True
7,gleam42b_2003_2020,216,2003-01-01 00:00:00,2020-12-01 00:00:00,,True
8,gracedadm_2003_2020,215,2003-02-01 00:00:00,2020-12-01 00:00:00,,True
9,merra2_1980_2020,492,1980-01-01 00:00:00,2020-12-01 00:00:00,,True
